# **Python y Google Earth Enginee**

## by **Juan C. Vega**

## *Univ. de los Andes*

**2026**

# Introducción

Este cuaderno presenta una introducción práctica al uso de **Google Earth Engine (GEE)** para el procesamiento, visualización y análisis de datos geoespaciales mediante Python. A través de la integración con librerías como `earthengine-api` y `geemap`, se construye un flujo de trabajo que permite acceder a grandes volúmenes de datos satelitales, procesarlos y visualizarlos de manera interactiva.

El enfoque principal se centra en el uso de imágenes satelitales provenientes de misiones como **Sentinel-2** y **Landsat 8**, las cuales permiten analizar variables ambientales clave como cobertura vegetal, temperatura superficial y características del terreno.

A lo largo del cuaderno se abordan los siguientes componentes:

- Autenticación e inicialización de Google Earth Engine desde Python.
- Exploración de datasets geoespaciales disponibles en la plataforma.
- Visualización interactiva de imágenes utilizando mapas dinámicos.
- Procesamiento de colecciones de imágenes, incluyendo reducción temporal (medianas).
- Cálculo de índices espectrales como el **NDVI (Normalized Difference Vegetation Index)** para análisis de vegetación.
- Ejemplos aplicados a diferentes regiones geográficas.

Este tipo de herramientas es fundamental en campos como la teledetección, geociencias, monitoreo ambiental y análisis de recursos naturales, permitiendo desarrollar soluciones basadas en datos a escala global.

El objetivo final es proporcionar una base sólida para construir flujos de trabajo más avanzados en análisis espacial, machine learning aplicado a imágenes satelitales y toma de decisiones basada en información geoespacial.

# Librería Geemap

## ¿Qué es Geemap?

`geemap` es una librería de Python que facilita la interacción con Google Earth Engine (GEE) mediante mapas interactivos dentro de entornos como Jupyter Notebook o Google Colab. Actúa como una capa de visualización que simplifica el uso de GEE sin necesidad de trabajar directamente con JavaScript.

## ¿Por qué usar Geemap?

Google Earth Engine está diseñado originalmente para ejecutarse en JavaScript desde su Code Editor. Sin embargo, en flujos de trabajo de ciencia de datos y machine learning, Python es el estándar. Aquí es donde `geemap` se vuelve clave.

Permite:
- Visualizar datos geoespaciales fácilmente
- Crear mapas interactivos sin esfuerzo
- Integrar análisis espacial con pipelines de Python

## Funcionalidades principales

### 1. Visualización interactiva

Permite desplegar mapas dinámicos donde se pueden superponer múltiples capas de información geoespacial.

```python
import geemap
Map = geemap.Map()
Map

**Integración con Earth Engine**

Se pueden agregar datasets directamente desde GEE:

`Map.addLayer(image, vis_params, "Nombre de la capa")`

**Herramientas de análisis**

Incluye utilidades para:

Dibujar geometrías
Extraer valores
Generar gráficos
Exportar datos

**Control de capas**

Permite activar/desactivar capas y comparar diferentes datasets fácilmente.

**Soporte para múltiples fuentes de datos**

No solo trabaja con GEE, también permite integrar:

GeoJSON
Shapefiles
Datos raster locales

# Metodología

El desarrollo de este cuaderno sigue una metodología estructurada para el análisis de datos geoespaciales utilizando Google Earth Engine (GEE) desde Python. El flujo de trabajo se divide en varias etapas clave:

## 1. Inicialización del entorno

Se establece la conexión con Google Earth Engine mediante autenticación del usuario y la inicialización de la API. Esto permite acceder a los datasets globales disponibles en la plataforma.

## 2. Selección de área de estudio (AOI)

Se define una región de interés (Area of Interest, AOI) utilizando coordenadas geográficas o geometrías. Esta área delimita el análisis espacial.

## 3. Selección de datos satelitales

Se accede a colecciones de imágenes como:
- Sentinel-2 (alta resolución espacial)
- Landsat 8 (análisis multitemporal)

Se aplican filtros por:
- Fecha
- Cobertura de nubes
- Área geográfica

## 4. Preprocesamiento de imágenes

Incluye:
- Filtrado de imágenes con alta nubosidad
- Recorte espacial al AOI
- Generación de composiciones temporales (por ejemplo, mediana)

## 5. Cálculo de índices espectrales

Se calculan métricas derivadas como el NDVI (Normalized Difference Vegetation Index), que permite evaluar la salud y densidad de la vegetación.

## 6. Visualización de resultados

Se utilizan mapas interactivos para:
- Mostrar imágenes satelitales
- Comparar resultados
- Explorar dinámicamente los datos

## 7. Análisis e interpretación

Se interpretan los resultados obtenidos, identificando patrones espaciales, variabilidad temporal y posibles aplicaciones en monitoreo ambiental.

---

Esta metodología permite estructurar análisis reproducibles y escalables, facilitando la integración con técnicas más avanzadas como machine learning o análisis predictivo.

In [ ]:
# Instala la API de Earth Engine y geemap para visualización
!pip install earthengine-api
!pip install geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 18.2 MB/s  0:00:00


In [ ]:
import ee

# Iniciar el proceso de autenticación
ee.Authenticate()

# Inicializar la librería con tu ID de proyecto específico que funciona correctamente
ee.Initialize(project='eloquent-ratio-462010-m4')

https://console.cloud.google.com/welcome?project=eloquent-ratio-462010-m4

In [ ]:
# probar conexion
import ee

# Referencia a una imagen de Landsat 8 (Colección 2 - Nivel 2)

# Probando con un conjunto de datos ampliamente accesible para verificar la conexión.
test_image = ee.Image('CGIAR/SRTM90_V4')

# Imprimir metadatos para verificar acceso
print(test_image.getInfo())

{'type': 'Image', 'bands': [{'id': 'elevation', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': -32768, 'max': 32767}, 'dimensions': [432000, 144000], 'crs': 'EPSG:4326', 'crs_transform': [0.000833333333333, 0, -180, 0, -0.000833333333333, 60]}], 'version': 1641990053291277, 'id': 'CGIAR/SRTM90_V4', 'properties': {'system:visualization_0_min': '-100.0', 'type_name': 'Image', 'keywords': ['cgiar', 'dem', 'elevation', 'geophysical', 'srtm', 'topography'], 'thumb': 'https://mw1.google.com/ges/dd/images/SRTM90_V4_thumb.png', 'description': '<p>The Shuttle Radar Topography Mission (SRTM) digital\nelevation dataset was originally produced to provide consistent,\nhigh-quality elevation data at near global scope. This version\nof the SRTM digital elevation data has been processed to fill data\nvoids, and to facilitate its ease of use.</p><p><b>Provider: <a href="https://srtm.csi.cgiar.org/">NASA/CGIAR</a></b><br><p><b>Bands</b><table class="eecat"><tr><th scope="col">Name</th><th

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import geopandas as gpd
kml_path = "/content/Valle_Piloto.kml"
gdf = gpd.read_file(kml_path, driver='KML')

DataSourceError: /content/Valle_Piloto.kml: No such file or directory

In [ ]:
# Guardar como GeoJSON
geojson_path = "/content/Valle_Piloto.geojson"
gdf.to_file(geojson_path, driver='GeoJSON')

In [ ]:
!pip install fiona

In [ ]:
!pip install fiona # Install fiona if not already installed

import geopandas as gpd
import fiona # Add this line to explicitly import fiona


# Ruta de entrada (KML)
kml_path = "/content/Valle_Piloto.kml"

# Leer KML (hay que habilitar el driver)
# Directly access fiona.drvsupport after importing fiona
fiona.drvsupport.supported_drivers['KML'] = 'rw'

gdf = gpd.read_file(kml_path, driver='KML')

# Verificar CRS (debería ser EPSG:4326 si es WGS84)
print(gdf.crs)

# Si no tiene CRS definido, lo asignas manualmente:
if gdf.crs is None:
    gdf.set_crs(epsg=4326, inplace=True)

# Guardar como Shapefile
shp_path = "/content/drive/MyDrive/GEE_EXPORT/Valle_Piloto.shp"
gdf.to_file(shp_path, driver='ESRI Shapefile')

print("Conversión completa.")

In [ ]:
import geopandas as gpd
import geemap
import ee
import json

# Inicializar Earth Engine con tu ID de proyecto
ee.Initialize(project='eloquent-ratio-462010-m4')

In [ ]:
# =========================
# RUTA SHAPEFILE
# =========================
shp_path = "/content/drive/MyDrive/GEE_EXPORT/Valle_Piloto.shp"

# =========================
# CARGAR Y LIMPIAR ROI
# =========================
roi = gpd.read_file(shp_path)

# Asegurar WGS84
roi = roi.to_crs(epsg=4326)

# Disolver en una sola geometría
roi = roi.dissolve()

# \ud83d\udd34 FIX CR\u00cdTICO: limpiar geometr\u00eda
roi["geometry"] = roi.geometry.buffer(0)

# DEBUG
print("Tipo:", roi.geometry.iloc[0].geom_type)
print("Bounds:", roi.total_bounds)

# =========================
# CONVERTIR A EE (MANUALMENTE DESDE COORDENADAS)
# =========================
# Extraer la geometría shapely limpia
shapely_polygon = roi.geometry.iloc[0]

# Extraer las coordenadas del exterior, asegurando formato [lon, lat]
exterior_coords = [[coord[0], coord[1]] for coord in shapely_polygon.exterior.coords]

# Extraer las coordenadas de los agujeros (interiores), si los hay, asegurando formato [lon, lat]
interior_coords = [[[coord[0], coord[1]] for coord in hole.coords] for hole in shapely_polygon.interiors]

# Construir ee.Geometry.Polygon directamente
if interior_coords:
    # Si hay agujeros, pasarlos como una lista de anillos (exterior + interiores)
    roi_ee = ee.Geometry.Polygon([exterior_coords] + interior_coords)
else:
    # Si no hay agujeros, pasar el exterior como una lista de un solo anillo
    roi_ee = ee.Geometry.Polygon([exterior_coords])

print("EE Geometry creado exitosamente.")

# =========================
# IMAGEN SRTM
# =========================
srtm = ee.Image("CGIAR/SRTM90_V4").clip(roi_ee)

# =========================
# MAPA
# =========================
Map = geemap.Map()

# Centrar en ROI
Map.centerObject(roi_ee, 9)

# Visualización
vis = {
    'bands': ['elevation'],
    'min': 0,
    'max': 6000,
    'palette': ['006633', 'E5FFCC', '662A00', 'D8D8D8', 'F5F5F5']
}

# Capas
Map.addLayer(srtm, vis, "Elevaci\u00f3n SRTM")
# Fix: Pass visualization parameters directly for ee.Geometry
Map.addLayer(
    roi_ee,
    {
        'color': 'red',
        'fillColor': '00000000',  # \ud83d\udd25 transparente
        'width': 2
    },
    "ROI"
)

# Control
Map.addLayerControl()

# MOSTRAR MAPA
Map

In [ ]:
import geopandas as gpd
import ee

roi = gpd.read_file(shp_path).to_crs(epsg=4326).dissolve()
roi["geometry"] = roi.geometry.buffer(0)

# Manual conversion to ee.Geometry to avoid 'Invalid GeoJSON geometry' error
shapely_polygon = roi.geometry.iloc[0]

# Extraer las coordenadas del exterior, asegurando formato [lon, lat]
exterior_coords = [[coord[0], coord[1]] for coord in shapely_polygon.exterior.coords]

# Extraer las coordenadas de los agujeros (interiores), si los hay, asegurando formato [lon, lat]
interior_coords = [[[coord[0], coord[1]] for coord in hole.coords] for hole in shapely_polygon.interiors]

# Construir ee.Geometry.Polygon directamente
if interior_coords:
    # Si hay agujeros, pasarlos como una lista de anillos (exterior + interiores)
    roi_ee = ee.Geometry.Polygon([exterior_coords] + interior_coords)
else:
    # Si no hay agujeros, pasar el exterior como una lista de un solo anillo
    roi_ee = ee.Geometry.Polygon([exterior_coords])

print("EE Geometry creado exitosamente (usando conversión manual).")

In [ ]:
# =========================
# IMAGEN SRTM
# =========================
test_image = ee.Image("CGIAR/SRTM90_V4")

# Recortar con ROI
srtm_clipped = test_image.clip(roi_ee)

# =========================
# CALCULAR RANGO REAL (CLAVE)
# =========================
stats = srtm_clipped.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi_ee,
    scale=90,
    bestEffort=True
)

stats_dict = stats.getInfo()
print("Min/Max:", stats_dict)

# Ajusta dinámicamente (fallback si falla)
min_val = stats_dict.get('elevation_min', 0)
max_val = stats_dict.get('elevation_max', 3000)

# =========================
# MAPA
# =========================
Map = geemap.Map()

Map.centerObject(roi_ee, 9)

# 🎨 VISUAL PRO
vis_params = {
    'bands': ['elevation'],
    'min': min_val,
    'max': max_val,
    'palette': [
        '0000ff',
        '00ffff',
        '00ff00',
        'ffff00',
        'ff0000',
        'ffffff'
    ]
}

# =========================
# CAPAS
# =========================
Map.addLayer(srtm_clipped, vis_params, 'Elevaci\u00f3n SRTM')

# 🔥 ROI SOLO BORDE (no relleno)
Map.addLayer(
    roi_ee,
    {
        'color': 'red',
        'fillColor': '00000000',  # 🔥 transparente
        'width': 2
    },
    'ROI'
)

Map.addLayerControl()

Map

In [ ]:
# =========================
# MAPA
# =========================
Map = geemap.Map()

Map.centerObject(roi_ee, 9)

# =========================
# COLECCIÓN SENTINEL-2
# =========================
coleccion = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(roi_ee) \
    .filterDate('2025-01-01', '2026-02-28') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))

# 🔴 DEBUG
print("Imágenes encontradas:", coleccion.size().getInfo())

# =========================
# MOSAICO (mediana para limpiar nubes residuales)
# =========================
imagen = coleccion.median().clip(roi_ee)

# =========================
# VISUAL RGB (natural color)
# =========================
vis_rgb = {
    'bands': ['B4', 'B3', 'B2'],  # rojo, verde, azul
    'min': 0,
    'max': 3000
}

# =========================
# CAPAS
# =========================
Map.addLayer(imagen, vis_rgb, 'Sentinel-2 RGB')

# ROI sin relleno
Map.addLayer(
    roi_ee,
    {
        'color': 'red',
        'fillColor': '00000000',
        'width': 2
    },
    'ROI'
)

Map.addLayerControl()

Map

In [ ]:
fechas = coleccion.aggregate_array('system:time_start').getInfo()

import datetime

fechas_legibles = [
    datetime.datetime.utcfromtimestamp(f/1000).strftime('%Y-%m-%d')
    for f in fechas
]

print(fechas_legibles)

# Procesamiento y Visualización

## **1. Reducción de Datos**

Como la colección tiene muchas imágenes, necesitamos convertirla en una sola para verla en el mapa. Usaremos la mediana para eliminar nubes residuales.


**2. Cálculo del Índice de Vegetación (NDVI)**

Usaremos la fórmula matemática para resaltar la salud de las plantas:
NDVI = NIR + Red/NIR − Red​

In [ ]:
# =========================
# MOSAICO
# =========================
imagen_final = coleccion.median().clip(roi_ee)

# =========================
# NDVI
# =========================
ndvi = imagen_final.normalizedDifference(['B8', 'B4']).rename('NDVI')

# =========================
# NDVI PROMEDIO EN ROI
# =========================
ndvi_mean = ndvi.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi_ee,
    scale=10,          # Sentinel-2 = 10 m
    bestEffort=True
)

ndvi_mean_val = ndvi_mean.get('NDVI').getInfo()
print("NDVI promedio en el ROI:", ndvi_mean_val)

# =========================
# MAPA
# =========================
Map = geemap.Map()

Map.centerObject(roi_ee, 10)

# RGB
vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000
}

Map.addLayer(imagen_final, vis_params, 'Imagen Satelital')

# NDVI
ndvi_params = {
    'min': 0,
    'max': 1,
    'palette': ['white', 'yellow', 'green']
}

Map.addLayer(ndvi, ndvi_params, 'NDVI')

# ROI sin relleno
Map.addLayer(
    roi_ee,
    {
        'color': 'red',
        'fillColor': '00000000',
        'width': 2
    },
    'ROI'
)

Map.addLayerControl()

Map

In [ ]:
# =========================
# FUNCION NDVI
# =========================
def add_ndvi(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return img.addBands(ndvi)

coleccion_ndvi = coleccion.map(add_ndvi)

# =========================
# LISTA DE MESES
# =========================
import pandas as pd

fechas = coleccion.aggregate_array('system:time_start').getInfo()

fechas_dt = pd.to_datetime([f for f in fechas], unit='ms')
meses = sorted(list(set(fechas_dt.strftime('%Y-%m'))))

print("Meses detectados:", meses)

# =========================
# NDVI MENSUAL
# =========================
ndvi_mensual_imgs = []
ndvi_mensual_vals = []

for mes in meses:
    inicio = ee.Date(f"{mes}-01")
    fin = inicio.advance(1, 'month')

    col_mes = coleccion_ndvi.filterDate(inicio, fin)

    # Promedio mensual
    ndvi_mes = col_mes.select('NDVI').mean().clip(roi_ee)

    # Calcular promedio espacial en ROI
    val = ndvi_mes.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi_ee,
        scale=10,
        bestEffort=True
    ).get('NDVI').getInfo()

    ndvi_mensual_imgs.append((mes, ndvi_mes))
    ndvi_mensual_vals.append((mes, val))

# =========================
# RESULTADOS
# =========================
print("\nNDVI promedio mensual:")
for mes, val in ndvi_mensual_vals:
    print(mes, ":", val)

# =========================
# MAPA (visualizar uno o varios meses)
# =========================
Map = geemap.Map()
Map.centerObject(roi_ee, 10)

vis_ndvi = {
    'min': 0,
    'max': 1,
    'palette': ['white', 'yellow', 'green']
}

# Mostrar algunos meses (ej: primeros 4)
for mes, img in ndvi_mensual_imgs[:4]:
    Map.addLayer(img, vis_ndvi, f"NDVI {mes}")

# ROI
Map.addLayer(
    roi_ee,
    {'color': 'red', 'fillColor': '00000000', 'width': 2},
    "ROI"
)

Map.addLayerControl()
Map

In [ ]:
# =========================
# FUNCION NDVI
# =========================
def add_ndvi(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return img.addBands(ndvi)

coleccion_ndvi = coleccion.map(add_ndvi)

# =========================
# EXTRAER MESES
# =========================
import pandas as pd

fechas = coleccion.aggregate_array('system:time_start').getInfo()
fechas_dt = pd.to_datetime(fechas, unit='ms')

meses = sorted(list(set(fechas_dt.strftime('%Y-%m'))))

# =========================
# CALCULAR NDVI MENSUAL
# =========================
resultados = []

for mes in meses:
    inicio = ee.Date(f"{mes}-01")
    fin = inicio.advance(1, 'month')

    col_mes = coleccion_ndvi.filterDate(inicio, fin)

    ndvi_mes = col_mes.select('NDVI').mean().clip(roi_ee)

    val = ndvi_mes.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi_ee,
        scale=10,
        bestEffort=True
    ).get('NDVI').getInfo()

    resultados.append({
        "fecha": mes,
        "ndvi": val
    })

# =========================
# DATAFRAME
# =========================
df = pd.DataFrame(resultados)

df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values("fecha")

print(df)

# =========================
# GRÁFICO
# =========================
import matplotlib.pyplot as plt

plt.figure()
plt.plot(df["fecha"], df["ndvi"], marker='o')
plt.title("NDVI mensual promedio en ROI")
plt.xlabel("Fecha")
plt.ylabel("NDVI")
plt.grid()

plt.show()

In [ ]:
# =========================
# NDVI POR IMAGEN
# =========================
def add_ndvi(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return img.addBands(ndvi)

coleccion_ndvi = coleccion.map(add_ndvi)

# =========================
# LISTA DE MESES
# =========================
import pandas as pd

fechas = coleccion.aggregate_array('system:time_start').getInfo()
fechas_dt = pd.to_datetime(fechas, unit='ms')

meses = sorted(list(set(fechas_dt.strftime('%Y-%m'))))

# Tomar últimos 6 meses
meses_sel = meses[-6:]

print("Meses a visualizar:", meses_sel)

# =========================
# GENERAR IMÁGENES NDVI
# =========================
ndvi_imgs = []

for mes in meses_sel:
    inicio = ee.Date(f"{mes}-01")
    fin = inicio.advance(1, 'month')

    col_mes = coleccion_ndvi.filterDate(inicio, fin)

    ndvi_mes = col_mes.select('NDVI').mean().clip(roi_ee)

    # Convertir a numpy para matplotlib
    img_np = geemap.ee_to_numpy(
        ndvi_mes,
        region=roi_ee,
        scale=10
    )

    ndvi_imgs.append((mes, img_np))

# =========================
# PLOTEAR SUBPLOTS
# =========================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes = axes.flatten()

for i, (mes, img) in enumerate(ndvi_imgs):
    ax = axes[i]

    # quitar dimensión extra
    img = img[:, :, 0]

    im = ax.imshow(img, vmin=0, vmax=1)
    ax.set_title(mes)
    ax.axis('off')

# barra de color
fig.colorbar(im, ax=axes, fraction=0.02)

plt.suptitle("NDVI mensual (últimos 6 meses)")
plt.tight_layout()

plt.show()

In [ ]:
# =========================
# LANDSAT 8 - TEMPERATURA
# =========================
l8_col = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(roi_ee) \
    .filterDate('2025-01-01', '2026-02-28') \
    .filter(ee.Filter.lt('CLOUD_COVER', 20))

# ✅ usar mediana
l8_img = l8_col.median().clip(roi_ee)

# Convertir ST_B10 (banda de temperatura superficial) a Celsius
# Formula: (Value * 0.00341802 + 149) - 273.15
temp_celsius = l8_img.select('ST_B10').multiply(0.00341802).add(149).subtract(273.15).rename('temp_celsius')

# =========================
# TEMPERATURA (°C)
# =========================
stats = temp_celsius.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi_ee,
    scale=30,
    bestEffort=True
)

print(stats.getInfo())

# =========================
# MAPA
# =========================
Map = geemap.Map()
Map.centerObject(roi_ee, 10)

Map.addLayer(
    temp_celsius,
    {
        'min': 32,
        'max': 45,
        'palette': ['blue', 'yellow', 'red']
    },
    'Temperatura Superficial'
)

Map.addLayer(
    roi_ee,
    {'color': 'red', 'fillColor': '00000000', 'width': 2},
    'ROI'
)

Map.addLayerControl()
Map

In [ ]:
# =========================
# LISTA DE MESES
# =========================
import pandas as pd

fechas = l8_col.aggregate_array('system:time_start').getInfo()
fechas_dt = pd.to_datetime(fechas, unit='ms')

meses = sorted(list(set(fechas_dt.strftime('%Y-%m'))))
meses_sel = meses[-6:]

# =========================
# GENERAR TEMPERATURA MENSUAL
# =========================
temp_imgs = []

for mes in meses_sel:
    inicio = ee.Date(f"{mes}-01")
    fin = inicio.advance(1, 'month')

    col_mes = l8_col.filterDate(inicio, fin)

    img = col_mes.median().clip(roi_ee) # This is the monthly median image with raw ST_B10

    # Convert ST_B10 to Celsius for the monthly image
    # Formula: (Value * 0.00341802 + 149) - 273.15
    temp_celsius_monthly = img.select('ST_B10').multiply(0.00341802).add(149).subtract(273.15)

    # The 'stats' block below was originally misplaced and likely referred to 'temp_celsius' from a previous cell.
    # It's removed from this loop to focus on monthly temperature processing.

    img_np = geemap.ee_to_numpy(
        temp_celsius_monthly, # Use the converted monthly temperature image
        region=roi_ee,
        scale=30   # Landsat = 30 m
    )

    temp_imgs.append((mes, img_np))

# =========================
# SUBPLOTS
# =========================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (mes, img_arr) in enumerate(temp_imgs): # Renamed `img` to `img_arr` to avoid confusion with ee.Image `img`
    ax = axes[i]

    img_arr = img_arr[:, :, 0]

    im = ax.imshow(img_arr, vmin=32, vmax=45)
    ax.set_title(mes)
    ax.axis('off')

fig.colorbar(im, ax=axes, fraction=0.02)

plt.suptitle("Temperatura superficial mensual (últimos 6 meses)")
plt.tight_layout()

plt.show()

In [ ]:
# Configurar exportacion a GEE

task = ee.batch.Export.image.toDrive(
    image=temp_celsius.clip(roi_ee),   # 🔥 recorte correcto
    description='Temperatura_ROI_2023',
    folder='GEE_Exports',
    fileNamePrefix='roi_temp_2023',
    region=roi_ee,          # 🔥 usar ROI real
    scale=30,                          # Landsat
    maxPixels=1e13,                    # 🔥 evita errores grandes
    fileFormat='GeoTIFF'
)

task.start()
print("Exportación iniciada. Revisa la pestaña 'Tasks' en el Code Editor o espera en Drive.")

# Resumen de tu aprendizaje:

* **Datos Ópticos (Sentinel-2):** Para ver la ciudad, vegetación y urbanismo.

* **Datos Térmicos (Landsat):** Para medir el calor y cambio climático local.

* **Exportación:** Conectas la potencia de procesamiento de Google con tu propio almacenamiento.

# **ENTRENAMIENTO SUPERVISADO**

## **1. setup - Carga de imagen**

In [ ]:
# =========================
# SENTINEL-2
# =========================
coleccion = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(roi_ee) \
    .filterDate('2025-01-01', '2026-02-28') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))

imagen = coleccion.median().clip(roi_ee)

## **2. Training - Markers y etiquetado**

### **2.1. Marker - Verde**


In [ ]:
Map_verde = geemap.Map()
Map_verde.centerObject(roi_ee, 12)
Map_verde.addLayer(imagen, {'bands': ['B4','B3','B2'], 'min':0, 'max':3000}, 'Dibujar VERDE')
Map_verde

### **2.1. Marker - Casas**

In [ ]:
Map_casas = geemap.Map()
Map_casas.centerObject(roi_ee, 12)
Map_casas.addLayer(imagen, {'bands': ['B4','B3','B2'], 'min':0, 'max':3000}, 'Dibujar CASAS')
Map_casas

## Etiqueta de Vias

In [ ]:
Map_Vegetacion = geemap.Map()
Map_Vegetacion.centerObject(roi_ee, 12)
Map_Vegetacion.addLayer(imagen, {'bands': ['B4','B3','B2'], 'min':0, 'max':3000}, 'Dibujar Vegetacion')
Map_Vegetacion

# **Crear Etiquetas**

In [ ]:
# =========================
# FUNCION PARA CLASE
# =========================
def set_class(fc, value):
    return fc.map(lambda f: f.set('class', value))

verde = set_class(Map_verde.user_rois, 0)
casas = set_class(Map_casas.user_rois, 1)
vias  = set_class(Map_Vegetacion.user_rois, 2)

# 🔥 IMPORTANTE: mejorar detección de vías
vias = vias.map(lambda f: f.buffer(20))

# Unir todo
training_points = verde.merge(casas).merge(vias)

print("Total puntos:", training_points.size().getInfo())

In [ ]:
# =========================
# INDICES
# =========================
ndvi = imagen.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndbi = imagen.normalizedDifference(['B11', 'B8']).rename('NDBI')

# Stack final
stack = imagen.select(['B2','B3','B4','B8']) \
    .addBands(ndvi) \
    .addBands(ndbi)

## **Entrenamiento**


In [ ]:
# =========================
# IMAGEN BASE
# =========================
imagen = coleccion.median().clip(roi_ee)

# =========================
# FEATURES
# =========================
ndvi = imagen.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndbi = imagen.normalizedDifference(['B11', 'B8']).rename('NDBI')

stack = imagen.select(['B2','B3','B4','B8','B11','B12']) \
    .addBands(ndvi) \
    .addBands(ndbi)

bandas_entrenamiento = stack.bandNames()

In [ ]:
# =========================
# SAMPLE
# =========================
training = stack.sampleRegions(
    collection=training_points,   # 👈 ya creado (verde, casas, vías)
    properties=['class'],
    scale=10
)

# =========================
# RANDOM FOREST
# =========================
classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=1000,
    minLeafPopulation=1
).train(
    features=training,
    classProperty='class',
    inputProperties=bandas_entrenamiento
)

In [ ]:
clasificado = stack.classify(classifier)

In [ ]:
Map2 = geemap.Map()
Map2.centerObject(roi_ee, 13)

# RGB base
Map2.addLayer(
    imagen,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'Imagen Real'
)

# 🔥 Paleta mejorada (3 clases)
vis_params = {
    'min': 0,
    'max': 2,
    'palette': [
        '#006400',  # 0 = Verde
        '#FFFFFF',  # 1 = Casas
        '#888888'   # 2 = Vías
    ]
}

clasificado_exists = False
try:
    # Attempt to get information about the classified image to check its validity.
    # This might throw the EEException if the image is invalid due to failed training.
    _ = clasificado.getInfo()
    print("Clasificación RF creada exitosamente.")
    Map2.addLayer(clasificado, vis_params, 'Clasificación RF')
    clasificado_exists = True
except ee.EEException as e:
    if "No valid training data were found." in str(e):
        print("Error: No se pudo generar la clasificación RF. No se encontraron datos de entrenamiento válidos.")
        print("Por favor, asegúrate de haber dibujado suficientes puntos para 'Verde', 'Casas' y 'Vías' en los mapas interactivos anteriores y de que estos puntos no caigan en áreas sin datos de la imagen Sentinel-2.")
    else:
        # Re-raise if it's a different EEException
        raise
except Exception as e:
    print(f"Error inesperado al verificar la clasificación RF: {e}")

if clasificado_exists:
    Map2.add_legend(
        title="Clasificación",
        labels={
            'Verde': '#006400',
            'Casas': '#FFFFFF',
            'Vías': '#888888'
        }
    )

# ROI limpio
Map2.addLayer(
    roi_ee,
    {'color': 'black', 'fillColor': '00000000', 'width': 2},
    'ROI'
)

if clasificado_exists:
    # Leyenda correcta (only add if classification was successful)
    Map2.add_legend(
        title="Clasificación",
        labels={
            'Verde': '#006400',
            'Casas': '#FFFFFF',
            'Vías': '#888888'
        }
    )

Map2.addLayerControl()
Map2

In [ ]:
# =========================
# IMAGEN BASE
# =========================
imagen = coleccion.median().clip(roi_ee)

# =========================
# ÍNDICES
# =========================
ndvi = imagen.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndbi = imagen.normalizedDifference(['B11', 'B8']).rename('NDBI')

# 🔥 textura (mejora para vías)
texture = imagen.select('B4').reduceNeighborhood(
    reducer=ee.Reducer.stdDev(),
    kernel=ee.Kernel.square(3)
).rename('texture')

# Stack final
stack = imagen.select(['B2','B3','B4','B8','B11','B12']) \
    .addBands(ndvi) \
    .addBands(ndbi) \
    .addBands(texture)

In [ ]:
# =========================
# BANDAS BASE
# =========================
bandas_base = ['B2', 'B3', 'B4', 'B8', 'B11']

# =========================
# ÍNDICES
# =========================
# 'imagen' is expected to be available from previous cells (e.g., from WRuEluyKumQL or 24K9BKnFzAu1)
ndvi = imagen.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndbi = imagen.normalizedDifference(['B11', 'B8']).rename('NDBI')
ndwi = imagen.normalizedDifference(['B3', 'B8']).rename('NDWI')

# 🔥 textura (importante para vías)
texture = imagen.select('B4').reduceNeighborhood(
    reducer=ee.Reducer.stdDev(),
    kernel=ee.Kernel.square(3)
).rename('texture')

# =========================
# STACK COMPLETO
# =========================
imagen_completa = imagen.select(bandas_base) \
    .addBands([ndvi, ndbi, ndwi, texture])

todas_las_bandas = bandas_base + ['NDVI', 'NDBI', 'NDWI', 'texture']


training = imagen_completa.sampleRegions(
    collection=training_points,
    properties=['class'],
    scale=10
)

classifier = ee.Classifier.smileRandomForest(1000).train(
    features=training,
    classProperty='class',
    inputProperties=todas_las_bandas
)

clasificado = imagen_completa.classify(classifier)

In [ ]:
ndwi = imagen.normalizedDifference(['B3', 'B8']).rename('NDWI')

stack = stack.addBands(ndwi)

In [ ]:
print(stack.bandNames().getInfo())

In [ ]:
clasificado = stack.classify(classifier)

In [ ]:
Map2 = geemap.Map()
Map2.centerObject(roi_ee, 13)

# Imagen base
Map2.addLayer(
    imagen,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'Imagen Real'
)

# 🔥 nueva paleta optimizada
vis_params = {
    'min': 0,
    'max': 2,
    'palette': [
        '#004d00',  # Verde oscuro
        '#FFFFFF',  # Casas (blanco)
        '#000000'   # Vías (negro)
    ],
    'opacity': 0.7
}

Map2.addLayer(clasificado, vis_params, 'Clasificación RF Mejorada')

# ROI limpio
Map2.addLayer(
    roi_ee,
    {'color': 'black', 'fillColor': '00000000', 'width': 2},
    'ROI'
)

Map2.add_legend(
    title="Clasificación",
    labels={
        'Verde': '#004d00',
        'Casas': '#FFFFFF',
        'Vías': '#000000'
    }
)

Map2.addLayerControl()
Map2

# **INTEGRACIoN TODA LA DATA**

In [ ]:
clasificado   # RF (0=verde,1=casas,2=vias)
ndvi          # NDVI imagen actual
temp_celsius  # temperatura Landsat

## 4. Preparación de la imagen con índices espectrales

Añadimos **NDVI**, **NDBI** y **NDWI** a las bandas originales.
Esto da a la red 9 canales (5 bandas base + 3 índices + 1 normalizado)
y mejora significativamente la separabilidad entre clases.

| Índice | Fórmula              | Útil para        |
|--------|----------------------|-----------------|
| NDVI   | (B8 - B4)/(B8 + B4)  | Vegetación       |
| NDBI   | (B11 - B8)/(B11+B8)  | Edificaciones    |
| NDWI   | (B3 - B8)/(B3 + B8)  | Agua / Piscinas  |

In [ ]:
# =========================
# BANDAS BASE
# =========================
bandas_base = ['B2', 'B3', 'B4', 'B8', 'B11']

# =========================
# ÍNDICES
# =========================
ndvi = imagen.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndbi = imagen.normalizedDifference(['B11', 'B8']).rename('NDBI')
ndwi = imagen.normalizedDifference(['B3', 'B8']).rename('NDWI')

# 🔥 textura (importante para vías)
texture = imagen.select('B4').reduceNeighborhood(
    reducer=ee.Reducer.stdDev(),
    kernel=ee.Kernel.square(3)
).rename('texture')

# =========================
# STACK COMPLETO
# =========================
imagen_completa = imagen.select(bandas_base) \
    .addBands([ndvi, ndbi, ndwi, texture])

todas_las_bandas = bandas_base + ['NDVI', 'NDBI', 'NDWI', 'texture']

print("Total bandas:", len(todas_las_bandas))
print("Bandas:", todas_las_bandas)

## 6. Exportar Parches GeoTIFF a Google Drive (Entrenamiento y Validación)

## 5. Preparación de imagen para exportación de parches con etiquetas

In [ ]:
# Rasterizar los training_points para crear una imagen de etiquetas
# Las clases deben estar representadas como valores de píxel (0, 1, 2, etc.).
labels_image = ee.Image.constant(0).uint8().paint(training_points, 'class').rename('label')

# Combina tu imagen de características (features) con la imagen de etiquetas.
# Asegúrate de que ambas estén clipadas a la misma ROI y tengan la misma resolución.
# 'imagen_completa' se definió anteriormente en la celda M5GuNhc89M1B.
image_to_export = imagen_completa.addBands(labels_image).clip(roi_ee)

# Opcional: Define los parámetros de visualización para la imagen de etiquetas si quieres verificarla en el mapa
# N_CLASSES se definió en e__hVaeI2Iwz
label_vis_params = {
    'min': 0,
    'max': N_CLASSES - 1,
    'palette': ['#006400', '#FFFFFF', '#000000'] # Verde, Casas, Vías (ajusta según tus clases)
}

# Crear un nuevo mapa para visualizar la imagen de etiquetas si es necesario
Map_labels = geemap.Map()
Map_labels.centerObject(roi_ee, 13)
Map_labels.addLayer(image_to_export.select(todas_las_bandas[:3]), {'min':0, 'max':3000}, 'Imagen Base (RGB)')
Map_labels.addLayer(labels_image, label_vis_params, 'Etiquetas de Clases')
Map_labels.addLayer(roi_ee, {'color': 'red', 'fillColor': '00000000', 'width': 2}, 'ROI')
Map_labels.addLayerControl()

print("Imagen para exportar (image_to_export) creada con éxito, incluyendo la banda 'label'.")
Map_labels # Muestra el mapa si estás en un entorno interactivo

In [ ]:
import random

# N_BANDS, N_CLASSES, PATCH_SIZE ya están definidos globalmente

# Rutas a las carpetas de entrenamiento y validación en Google Drive
# Asegúrate de que estas carpetas existan en tu Google Drive manualmente.
TRAIN_FOLDER = 'GEE_EXPORT/unet_patches/train_patches'
VAL_FOLDER   = 'GEE_EXPORT/unet_patches/val_patches'

# Definir el tamaño de la ventana de exportación
# GRID_SIZE debe ser un múltiplo de PATCH_SIZE y del stride de la U-Net si se usa Sliding Window
# Para simplicidad y eficiencia, usamos PATCH_SIZE para el tamaño del parche a exportar

# Obtenemos las coordenadas del bounding box de la ROI
bbox = roi_ee.bounds()

# Calculamos el número de parches que caben en la ROI
# Esto es un ejemplo, se puede ajustar la estrategia de cuadriculado
coords = bbox.getInfo()['coordinates'][0]
min_lon, min_lat = coords[0]
max_lon, max_lat = coords[2]

# Generar una cuadrícula de puntos para centrar los parches
# La escala de Sentinel-2 es 10m, así que usaremos 10m para la exportación.
# Para una ventana de 64x64, la distancia entre centros de parche podría ser 640m
# o un stride menor para solapamiento.
# Aquí se usa un stride de PATCH_SIZE * scale (640m) para evitar solapamiento excesivo.

# Esta es una aproximación, para una cobertura completa y sin huecos se necesitaría
# un algoritmo de cuadriculado más robusto o el enfoque de ee.Image.sampleRegions
# sobre un grid de puntos.

# Para este ejemplo, vamos a exportar un número fijo de parches aleatorios o
# un grid simple para demostrar la exportación.

# --- Estrategia de Grid Simple para demostración ---
# Vamos a generar un grid de puntos sobre el roi_ee y usar cada punto como centro de un parche.

def create_grid_points(geometry, patch_dim_m, spacing_m):
    """Crea una FeatureCollection de puntos en una cuadrícula dentro de una geometría.
       patch_dim_m: dimensión del parche en metros (ej. 64 * 10m = 640m)
       spacing_m: espaciado entre centros de parches en metros.
    """
    # Obtener el bounding box de la geometría en el CRS proyectado para trabajar con metros
    projected_geom = geometry.transform('EPSG:3857', 1) # Proyectar a un CRS métrico
    # 🔥 FIX: Añadir errorMargin a bounds()
    proj_bounds = projected_geom.bounds(ee.ErrorMargin(1)).getInfo()['coordinates'][0]

    # Extraer las coordenadas x e y de las esquinas del bounding box
    minX = proj_bounds[0][0]
    minY = proj_bounds[0][1]
    maxX = proj_bounds[2][0]
    maxY = proj_bounds[2][1]

    # Calcular el rango válido para los centros de los parches
    # Los centros deben estar lo suficientemente alejados de los bordes para que el parche completo caiga dentro de los límites.
    center_minX = minX + patch_dim_m / 2
    center_maxX = maxX - patch_dim_m / 2
    center_minY = minY + patch_dim_m / 2
    center_maxY = maxY - patch_dim_m / 2

    # Generar secuencias para las coordenadas x e y de los centros de los parches usando ee.List.sequence
    # Esto asegura el manejo correcto de flotantes y rangos.
    # Añadir una verificación para asegurar que el rango es válido antes de crear la secuencia
    x_coords = ee.List([])
    y_coords = ee.List([])

    if center_maxX > center_minX:
        x_coords = ee.List.sequence(center_minX, center_maxX, spacing_m)
    if center_maxY > center_minY:
        y_coords = ee.List.sequence(center_minY, center_maxY, spacing_m)

    # Si alguna de las listas de coordenadas está vacía, no hay puntos para generar
    if x_coords.size().getInfo() == 0 or y_coords.size().getInfo() == 0:
        return ee.FeatureCollection([])

    # Crear una lista de puntos a partir de las secuencias de centros x e y
    # Transformar cada punto de nuevo a EPSG:4326 para consistencia
    points_list = x_coords.map(lambda x: y_coords.map(
        lambda y: ee.Geometry.Point([x, y], 'EPSG:3857').transform('EPSG:4326', 1)
    )).flatten()

    # Convertir la lista de puntos a una FeatureCollection
    grid_points_fc = ee.FeatureCollection(points_list.map(lambda p: ee.Feature(p)))

    # Filtrar para mantener solo los puntos que caen dentro de la ROI original.
    # Esto sirve como una salvaguarda final contra puntos generados fuera de ROIs complejas.
    filtered_grid = grid_points_fc.filterBounds(geometry)

    return filtered_grid

# Parámetros del grid
PATCH_DIM_METERS = PATCH_SIZE * 10  # 64 píxeles * 10 metros/píxel = 640 metros
SPACING_METERS   = PATCH_DIM_METERS // 2 # solapamiento del 50%

grid_points = create_grid_points(roi_ee, PATCH_DIM_METERS, SPACING_METERS)
grid_size = grid_points.size().getInfo() # Obtener el tamaño de la colección una vez

print(f"Se generaron {grid_size} puntos de cuadrícula.")

regions_to_export = ee.List([]) # Inicializar como lista vacía

# Convertir FeatureCollection a una lista de geometrías (regiones para exportar) SOLO SI hay puntos
if grid_size > 0:
    regions_to_export = grid_points.toList(grid_size)

# Dividir los parches en entrenamiento y validación
num_patches = regions_to_export.size().getInfo() # Esto funcionará para una lista vacía
if num_patches == 0:
    print("No hay parches para exportar. Verifica tu ROI y la configuración del grid. Considera reducir PATCH_DIM_METERS o SPACING_METERS.")
else:
    train_split_ratio = 0.8
    num_train_patches = int(num_patches * train_split_ratio)

    # Randomizar la lista para una división aleatoria
    random_indices = list(range(num_patches))
    random.shuffle(random_indices)

    for i, idx in enumerate(random_indices):
        patch_region = ee.Geometry(regions_to_export.get(idx)).buffer(PATCH_DIM_METERS / 2)

        # Obtener la clase predominante en este parche (para el nombre del archivo)
        # Esto es crucial para que el dataset PyTorch pueda leer las etiquetas.
        dominant_label_reducer = labels_image.reduceRegion(
            reducer=ee.Reducer.mode(), # Moda para la clase predominante
            geometry=patch_region,
            scale=10, # Resolución de Sentinel-2
            maxPixels=1e9,
            tileScale=16
        )
        dominant_label = dominant_label_reducer.get('label').getInfo()
        if dominant_label is None:
             # Si no hay píxeles etiquetados en el parche, asigna una clase por defecto o ignora
            # Aquí, asignamos -1 o podemos decidir no exportar este parche.
            # Para este ejemplo, si no hay label, se asigna 0 (Verde) como fallback.
            dominant_label = 0
            print(f"Advertencia: No se encontró etiqueta dominante para el parche {i}, asignando 0.")

        # Decidir si es entrenamiento o validación
        if i < num_train_patches:
            target_folder = TRAIN_FOLDER
        else:
            target_folder = VAL_FOLDER

        # Crear un nombre de archivo único con la clase y un identificador
        file_name_prefix = f'patch_cls{dominant_label}_id{idx}'

        # Iniciar la tarea de exportación
        task = ee.batch.Export.image.toDrive(
            image=image_to_export, # La imagen que contiene las bandas y la etiqueta
            description=f'Export_Patch_{file_name_prefix}',
            folder=target_folder, # Carpeta en Google Drive
            fileNamePrefix=file_name_prefix,
            region=patch_region, # La región de interés para este parche
            scale=10,            # Resolución de Sentinel-2
            maxPixels=1e9,       # Evitar errores de maxPixels
            fileFormat='GeoTIFF',
            formatOptions={
                'cloudOptimized': True # Optimizar para lectura en la nube
            }
        )
        task.start()
        print(f"Exportando parche {i+1}/{num_patches} (Clase {dominant_label}) a {target_folder} como {file_name_prefix}.tif")

    print("\nTodas las tareas de exportación han sido iniciadas. Revisa la pestaña 'Tasks' en el Code Editor de Google Earth Engine para su progreso.")
    print("Una vez que las exportaciones se completen, podrás ejecutar las celdas para el entrenamiento de la U-Net.")

In [ ]:
import os
import rasterio
from rasterio.mask import mask
import numpy as np
import pandas as pd

ruta_base = "/content/drive/MyDrive/GEE_EXPORT/Humedad_mensual"

# =========================
# 1. BUSCAR TODAS LAS CARPETAS -M (RECURSIVO 🔥)
# =========================
carpetas_M = []

for root, dirs, files in os.walk(ruta_base):
    for d in dirs:
        if d.strip().endswith("-M"):
            carpetas_M.append(os.path.join(root, d))

print("Carpetas detectadas:")
for c in carpetas_M:
    print(c)

# =========================
# 2. EXTRAER TODOS LOS TIFF
# =========================
archivos_tif = []

for carpeta in carpetas_M:
    for f in os.listdir(carpeta):
        if f.endswith(".tif"):
            archivos_tif.append({
                "variable": os.path.basename(carpeta),  # 🔥 nombre limpio
                "archivo": f,
                "path": os.path.join(carpeta, f)
            })

print("Total archivos encontrados:", len(archivos_tif))

# =========================
# 3. ROI
# =========================
roi_geom = roi.geometry.values[0]

# =========================
# 4. EXTRACCIÓN
# =========================
resultados = []

for item in archivos_tif:
    try:
        with rasterio.open(item["path"]) as src:
            out_image, _ = mask(src, [roi_geom], crop=True)

            data = out_image[0]
            data = data[data != src.nodata]

            resultados.append({
                "variable": item["variable"],
                "archivo": item["archivo"],
                "valor": np.nanmean(data)
            })
    except Exception as e:
        print("Error en:", item["path"], e)

# =========================
# 5. DATAFRAME
# =========================
df_wapor = pd.DataFrame(resultados)

print("\ndf_wapor creado ✔")
print(df_wapor.head())

# Check if 'variable' column exists before trying to access it
if "variable" in df_wapor.columns:
    print("\nVariables detectadas:", df_wapor["variable"].unique())
elif df_wapor.empty:
    print("\nNo se detectaron variables porque df_wapor está vacío.")
    print("Por favor, verifica la ruta de los archivos en Google Drive (`ruta_base`).")
else:
    print("\nNo se detectó la columna 'variable' en df_wapor.")

In [ ]:
import pandas as pd

# =========================
# EXTRAER FECHA CORRECTAMENTE
# =========================
# Verifica si df_wapor no está vacío antes de intentar procesarlo
if not df_wapor.empty:
    # Extract year and month into temporary columns
    df_wapor[['year', 'month']] = df_wapor["archivo"].str.extract(r'(\d{4})_(\d{2})')

    # Combine year and month into a single 'fecha' string
    df_wapor["fecha"] = df_wapor.apply(
        lambda x: f"{x['year']}-{x['month']}" if pd.notnull(x['year']) else None,
        axis=1
    )

    # Convert to datetime, coercing errors to NaT
    df_wapor["fecha"] = pd.to_datetime(df_wapor["fecha"], errors='coerce')

    # Drop temporary columns and rows with bad dates
    df_wapor = df_wapor.drop(columns=['year', 'month']).dropna(subset=["fecha"])

    print(df_wapor.head(50))
else:
    print("df_wapor está vacío. No se puede extraer la fecha. Por favor, verifica que los archivos .tif existen en la ruta de Google Drive y que la celda anterior se ejecutó correctamente.")


In [ ]:
df_pivot = df_wapor.pivot_table(
    index="fecha",
    columns="variable",
    values="valor",
    aggfunc="mean"
).reset_index()

In [ ]:
df_final = df.merge(df_pivot, on="fecha", how="inner")

print(df_final.head())

In [ ]:
print("NDVI fechas:")
print(df["fecha"].min(), df["fecha"].max())

print("\WaPOR fechas:")
print(df_pivot["fecha"].min(), df_pivot["fecha"].max())

In [ ]:
import matplotlib.pyplot as plt

plt.figure()

plt.plot(df_final["fecha"], df_final["ndvi"], label="NDVI")

if "L2-RSM-M" in df_final.columns:
    plt.plot(df_final["fecha"], df_final["L2-RSM-M"], label="RSM")

plt.legend()
plt.title("NDVI vs Humedad (WaPOR mensual)")
plt.xticks(rotation=45)
plt.grid()

plt.show()

# ─── DATASET PYTORCH ──────────────────────────────────────────────────────────

# 🧠 MÓDULO U-NET — Clasificación con Deep Learning

# 6. Dataset PyTorch para parches GeoTIFF



In [ ]:
import os, glob, re
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split

try:
    import rasterio
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False
    print("⚠️  rasterio no disponible — instalando ...")
    os.system("pip install rasterio --quiet")
    import rasterio
    HAS_RASTERIO = True

# ── Configuración global ──────────────────────────────────────────────────────
N_CLASSES   = 3          # 0=Verde, 1=Casas, 2=Piscinas
N_BANDS     = 8          # B2 B3 B4 B8 B11 NDVI NDBI NDWI
PATCH_SIZE  = 64
BATCH_SIZE  = 8
SEED        = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# Dataset ────────────────────────────────────────────────


In [ ]:
class SentinelPatchDataset(Dataset):

    def __init__(self, tif_dir, patch_size=PATCH_SIZE, augment=False):
        self.files    = sorted(glob.glob(os.path.join(tif_dir, '*.tif')))
        self.psize    = patch_size
        self.augment  = augment
        if len(self.files) == 0:
            raise FileNotFoundError(f"No se encontraron archivos .tif en {tif_dir}")

    def __len__(self):
        return len(self.files)

    def _extract_label(self, path):
        m = re.search(r'cls(\d+)', os.path.basename(path))
        return int(m.group(1)) if m else 0

    def _make_mask(self, label, h, w):
        """Máscara uniforme con la clase del parche."""
        return torch.full((h, w), label, dtype=torch.long)

    def _read_patch(self, path):
        with rasterio.open(path) as src:
            arr = src.read().astype(np.float32)   # (C, H, W)
        # Normalización min-max por banda
        for b in range(arr.shape[0]):
            mn, mx = arr[b].min(), arr[b].max()
            arr[b] = (arr[b] - mn) / (mx - mn + 1e-8)
        return arr

    def __getitem__(self, idx):
        path  = self.files[idx]
        label = self._extract_label(path)
        arr   = self._read_patch(path)       # (C, H, W)

        # Recortar / rellenar al tamaño deseado
        c, h, w = arr.shape
        ph = min(h, self.psize);  pw = min(w, self.psize)
        patch = np.zeros((c, self.psize, self.psize), dtype=np.float32)
        patch[:, :ph, :pw] = arr[:, :ph, :pw]

        X = torch.from_numpy(patch)           # (C, P, P)
        Y = self._make_mask(label, self.psize, self.psize)  # (P, P)

        if self.augment:
            if torch.rand(1) > 0.5:
                X = torch.flip(X, dims=[2])
                Y = torch.flip(Y, dims=[1])
            if torch.rand(1) > 0.5:
                X = torch.flip(X, dims=[1])
                Y = torch.flip(Y, dims=[0])

        return X, Y

# ─── U-NET ────────────────────────────

## Arquitectura U-Net — Clasificación Semántica Pixel a Pixel

La U-Net es la arquitectura de referencia para segmentación semántica de imágenes
de teledetección. A diferencia del Random Forest (que opera píxel a píxel),
la U-Net captura **contexto espacial** en su encoder y lo recupera con precisión
gracias a las **skip connections** que conectan encoder y decoder.

```
Entrada (8ch, 64×64)
   │
   ├── enc1 (32ch)  ──────────────────────────────────────┐ skip
   │    └── pool                                          │
   │         ├── enc2 (64ch)  ──────────────────────┐ skip│
   │         │    └── pool                          │    │
   │         │         ├── enc3 (128ch)  ──────┐ skip│    │
   │         │         │    └── pool           │    │    │
   │         │         │         └── enc4 (256ch) ─┐│    │
   │         │         │              └── pool     ││    │
   │         │         │                   └── bottleneck (512ch)
   │         │         │              up4 ──────────┘│    │
   │         │         │         dec4 (256ch) cat────┘    │
   │         │         │    up3                           │
   │         │         dec3 (128ch) cat──────────────┘    │
   │         │    up2                                     │
   │         dec2 (64ch) cat──────────────────────────┘   │
   │    up1                                               │
   dec1 (32ch) cat────────────────────────────────────────┘
   │
   out_conv (3ch) → Softmax → Máscara [0=Verde, 1=Casas, 2=Piscinas]
```


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    """Bloque doble Conv → BN → ReLU con Dropout espacial."""
    def __init__(self, in_ch, out_ch, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """
    U-Net multiclase para clasificación semántica de imágenes satelitales.

    Parámetros
    ----------
    in_channels : int
        Número de bandas de entrada (default: 8 — Sentinel-2 + índices).
    base_ch     : int
        Canales base (se duplican en cada nivel del encoder).
    n_classes   : int
        Número de clases de salida (default: 3 — Verde, Casas, Piscinas).
    dropout     : float
        Tasa de Dropout2d en cada ConvBlock.
    """
    def __init__(self, in_channels=8, base_ch=32, n_classes=3, dropout=0.1):
        super().__init__()
        bc = base_ch

        # ── Encoder ────────────────────────────────────
        self.enc1 = ConvBlock(in_channels, bc,    dropout)   # → bc
        self.enc2 = ConvBlock(bc,          bc*2,  dropout)   # → bc*2
        self.enc3 = ConvBlock(bc*2,        bc*4,  dropout)   # → bc*4
        self.enc4 = ConvBlock(bc*4,        bc*8,  dropout)   # → bc*8
        self.pool = nn.MaxPool2d(2)

        # ── Bottleneck ─────────────────────────
        self.bottleneck = ConvBlock(bc*8, bc*16, dropout)    # → bc*16

        # ── Decoder ─────────────────────────────────
        self.up4  = nn.ConvTranspose2d(bc*16, bc*8,  2, stride=2)
        self.dec4 = ConvBlock(bc*16, bc*8,  dropout)

        self.up3  = nn.ConvTranspose2d(bc*8,  bc*4,  2, stride=2)
        self.dec3 = ConvBlock(bc*8,  bc*4,  dropout)

        self.up2  = nn.ConvTranspose2d(bc*4,  bc*2,  2, stride=2)
        self.dec2 = ConvBlock(bc*4,  bc*2,  dropout)

        self.up1  = nn.ConvTranspose2d(bc*2,  bc,    2, stride=2)
        self.dec1 = ConvBlock(bc*2,  bc,    dropout)

        # ── Cabeza de salida — N clases ─────────────────────
        self.out_conv = nn.Conv2d(bc, n_classes, kernel_size=1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias,   0)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b  = self.bottleneck(self.pool(e4))

        # Decoder con skip connections
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        # Logits (N, n_classes, H, W) — sin softmax para usar CrossEntropy
        return self.out_conv(d1)


# ── Verificación rápida ──────────────────────────────────────
model_test = UNet(in_channels=N_BANDS, base_ch=32, n_classes=N_CLASSES)
total_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {total_params:,}")

dummy = torch.randn(2, N_BANDS, PATCH_SIZE, PATCH_SIZE)
out   = model_test(dummy)
print(f"Input:  {tuple(dummy.shape)}  →  Output: {tuple(out.shape)}")
print(f"Esperado: [2, {N_CLASSES}, {PATCH_SIZE}, {PATCH_SIZE}]  ✅")


# ─── FUNCIÓN DE PÉRDIDA ───────────────────

## Función de pérdida combinada — CrossEntropy + Dice Multiclase

Para clasificación multiclase utilizamos:

- **CrossEntropy:** Penaliza errores de clasificación por clase.
- **Dice Multiclase:** Penaliza el desequilibrio de clases (clave para detectar
  piscinas, que son mucho menos frecuentes que vegetación o edificios).


In [ ]:
import torch
import torch.nn.functional as F


def dice_loss_multiclass(logits, targets, n_classes=N_CLASSES, smooth=1.0):
    """
    Dice Loss promediado sobre todas las clases.
    logits  : (N, C, H, W) — salida cruda de la red
    targets : (N, H, W)    — etiquetas enteras [0..C-1]
    """
    probs = F.softmax(logits, dim=1)   # (N, C, H, W)
    loss  = 0.0
    for cls in range(n_classes):
        pred_cls   = probs[:, cls].reshape(-1)
        target_cls = (targets == cls).float().reshape(-1)
        intersection = (pred_cls * target_cls).sum()
        loss += 1 - (2 * intersection + smooth) / (
                    pred_cls.sum() + target_cls.sum() + smooth)
    return loss / n_classes


def combined_loss(logits, targets, alpha=0.5):
    """α · CrossEntropy + (1-α) · Dice multiclase."""
    ce   = F.cross_entropy(logits, targets)
    dice = dice_loss_multiclass(logits, targets)
    return alpha * ce + (1 - alpha) * dice


print("✅ Funciones de pérdida definidas: CrossEntropy + Dice Multiclase (α=0.5)")


# ─── ENTRENAMIENTO ───────────────────────
cells.append(md_cell("""\
## 9. Entrenamiento de la U-Net

- **Optimizador:** AdamW con decaimiento de pesos (`weight_decay=1e-4`)
- **Scheduler:** Cosine Annealing para suavizar el descenso del learning rate
- **Épocas:** 50 (reducir si usas CPU; aumentar con GPU para mejores resultados)

In [ ]:
import torch
import os
from torch.utils.data import DataLoader # Import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

model     = UNet(in_channels=N_BANDS, base_ch=32, n_classes=N_CLASSES).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS    = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

history = {'train': [], 'val': []}

# TODO: Asegúrate de que las rutas a los parches de entrenamiento y validación sean correctas.
# Estos directorios deben contener los archivos GeoTIFF exportados desde GEE.
train_dir = "/content/drive/MyDrive/GEE_EXPORT/unet_patches/train_patches" # <--- ¡ACTUALIZA ESTA RUTA SI ES NECESARIO!
val_dir   = "/content/drive/MyDrive/GEE_EXPORT/unet_patches/val_patches"   # <--- ¡ACTUALIZA ESTA RUTA SI ES NECESARIO!

# Crear datasets
train_dataset = SentinelPatchDataset(train_dir, patch_size=PATCH_SIZE, augment=True)
val_dataset   = SentinelPatchDataset(val_dir,   patch_size=PATCH_SIZE, augment=False)

# Crear DataLoaders
dl_train = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=os.cpu_count(), pin_memory=True)
dl_val   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=os.cpu_count(), pin_memory=True)

print(f"Dataset de entrenamiento: {len(train_dataset)} parches")
print(f"Dataset de validación: {len(val_dataset)} parches")

for epoch in range(1, EPOCHS + 1):
    # ── Entrenamiento ────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for X, Y in dl_train:
        X, Y = X.to(device), Y.to(device)
        logits = model(X)
        loss   = combined_loss(logits, Y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(dl_train)


In [ ]:
# ── Validación ──────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X, Y in dl_val:
            X, Y = X.to(device), Y.to(device)
            val_loss += combined_loss(model(X), Y).item()
    val_loss /= len(dl_val)

    scheduler.step()
    history['train'].append(train_loss)
    history['val'].append(val_loss)

    if epoch % 5 == 0 or epoch == 1 or epoch == EPOCHS:
        print(f"Epoch {epoch:3d}/{EPOCHS}  |  "
              f"Train: {train_loss:.4f}  |  Val: {val_loss:.4f}  |  "
              f"LR: {scheduler.get_last_lr()[0]:.2e}")

print("\\n✅ Entrenamiento completado.")


In [ ]:
# ─── CURVAS ───────────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history['train'], label='Train Loss', color='steelblue', lw=2)
ax.plot(history['val'],   label='Val Loss',   color='tomato',    lw=2)
ax.set_xlabel('Época')
ax.set_ylabel('Loss (CE + Dice)')
ax.set_title('Curvas de Aprendizaje — U-Net Multiclase (3 clases)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('unet_training_curves.png', dpi=150)
plt.show()
print("Figura guardada: unet_training_curves.png")

In [ ]:
# ─── MÉTRICAS ─────────────────────────────────────────────────────────────────

import torch
import numpy as np

def compute_iou_f1(model, dataloader, n_classes=N_CLASSES, device=device):
    model.eval()
    confusion = np.zeros((n_classes, n_classes), dtype=np.int64)
    with torch.no_grad():
        for X, Y in dataloader:
            X, Y = X.to(device), Y.to(device)
            preds = model(X).argmax(dim=1)   # (N, H, W)
            for t, p in zip(Y.cpu().numpy().ravel(), preds.cpu().numpy().ravel()):
                confusion[t, p] += 1

    ious, f1s = [], []
    names = ['Vegetación (0)', 'Casas (1)', 'Piscinas (2)']
    print(f"{'Clase':<20} {'IoU':>8} {'F1':>8} {'Precisión':>12} {'Recall':>10}")
    print("-" * 62)
    for cls in range(n_classes):
        tp = confusion[cls, cls]
        fp = confusion[:, cls].sum() - tp
        fn = confusion[cls, :].sum() - tp
        iou = tp / (tp + fp + fn + 1e-8)
        pre = tp / (tp + fp + 1e-8)
        rec = tp / (tp + fn + 1e-8)
        f1  = 2 * pre * rec / (pre + rec + 1e-8)
        ious.append(iou);  f1s.append(f1)
        print(f"{names[cls]:<20} {iou:>8.3f} {f1:>8.3f} {pre:>12.3f} {rec:>10.3f}")
    print("-" * 62)
    print(f"{'Mean IoU':<20} {np.mean(ious):>8.3f}")
    print(f"{'Mean F1':<20} {np.mean(f1s):>8.3f}")
    return confusion

confusion_matrix = compute_iou_f1(model, dl_val)


In [ ]:
# ─── VISUALIZACIÓN DE PREDICCIONES ────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch

CLASS_COLORS = np.array([
    [0,   180,  0],    # 0 — Vegetación (verde)
    [220,  50, 50],    # 1 — Casas      (rojo)
    [30,  100, 220],   # 2 — Piscinas   (azul)
], dtype=np.uint8)

def mask_to_rgb(mask_np):
    rgb = CLASS_COLORS[mask_np]  # (H, W, 3)
    return rgb

model.eval()
X_batch, Y_batch = next(iter(dl_val))
X_batch = X_batch.to(device)

with torch.no_grad():
    pred_batch = model(X_batch).argmax(dim=1).cpu().numpy()

Y_batch = Y_batch.numpy()
X_show  = X_batch.cpu().numpy()

n_show = min(4, len(X_batch))
fig, axes = plt.subplots(n_show, 3, figsize=(12, 3 * n_show))
if n_show == 1:
    axes = axes[np.newaxis]

for i in range(n_show):
    # RGB falso: bandas 2, 1, 0 (B4, B3, B2)
    rgb = X_show[i, [2, 1, 0]].transpose(1, 2, 0)
    rgb = np.clip((rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8), 0, 1)

    axes[i, 0].imshow(rgb)
    axes[i, 0].set_title('Imagen (Color Real)');  axes[i, 0].axis('off')

    axes[i, 1].imshow(mask_to_rgb(Y_batch[i]))
    axes[i, 1].set_title('Máscara Real');          axes[i, 1].axis('off')

    axes[i, 2].imshow(mask_to_rgb(pred_batch[i]))
    axes[i, 2].set_title('Predicción U-Net');      axes[i, 2].axis('off')

legend_patches = [
    mpatches.Patch(color=[c/255 for c in CLASS_COLORS[0]], label='Vegetación (0)'),
    mpatches.Patch(color=[c/255 for c in CLASS_COLORS[1]], label='Casas (1)'),
    mpatches.Patch(color=[c/255 for c in CLASS_COLORS[2]], label='Piscinas (2)'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=11,
           bbox_to_anchor=(0.5, -0.04))
plt.suptitle('U-Net — Comparación: Imagen Real vs Máscara vs Predicción', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('unet_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: unet_predictions.png")

In [ ]:
# ─── GUARDAR MODELO ───────────────────────────────────────────────────────────

import torch, os

SAVE_PATH = '/content/drive/MyDrive/GEE_EXPORT/unet_valledupar_3clases.pth'

torch.save({
    'model_state_dict': model.state_dict(),
    'in_channels': N_BANDS,
    'n_classes':   N_CLASSES,
    'patch_size':  PATCH_SIZE,
    'history':     history,
}, SAVE_PATH)

print(f"✅ Modelo guardado en: {SAVE_PATH}")
print("   Para cargarlo en otra sesión:")
print(\"   checkpoint = torch.load(SAVE_PATH)\")
print(\"   model.load_state_dict(checkpoint['model_state_dict'])\")

# ─── INFERENCIA GEE ──────────────────────────────

## 14. Inferencia sobre imagen completa exportada desde GEE

Para aplicar la U-Net a una imagen completa (no parches de entrenamiento)
necesitamos una **estrategia de ventana deslizante** (*sliding window*).

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

def predict_image_sliding_window(model, image_np, patch_size=PATCH_SIZE,
                                  stride=32, n_classes=N_CLASSES, device=device):
    \"\"\"
    Aplica la U-Net a una imagen completa usando ventana deslizante.

    Parámetros
    ----------
    image_np : np.ndarray  (C, H, W)  — imagen normalizada [0, 1]
    \"\"\"
    model.eval()
    C, H, W = image_np.shape
    prob_map = np.zeros((n_classes, H, W), dtype=np.float32)
    count    = np.zeros((H, W),           dtype=np.float32)

    with torch.no_grad():
        for y in range(0, H - patch_size + 1, stride):
            for x in range(0, W - patch_size + 1, stride):
                patch = image_np[:, y:y+patch_size, x:x+patch_size]
                t     = torch.from_numpy(patch).unsqueeze(0).to(device)   # (1,C,P,P)
                logit = model(t)                                           # (1,Cl,P,P)
                prob  = F.softmax(logit, dim=1).squeeze(0).cpu().numpy()  # (Cl,P,P)
                prob_map[:, y:y+patch_size, x:x+patch_size] += prob
                count[y:y+patch_size, x:x+patch_size]       += 1

    count = np.maximum(count, 1)
    pred  = (prob_map / count).argmax(axis=0).astype(np.uint8)  # (H, W)
    return pred


print("✅ Función de inferencia por ventana deslizante lista.")
print("   Uso:")
print("   >>> pred_mask = predict_image_sliding_window(model, image_np)")
print("   image_np debe ser un array (C, H, W) con C=N_BANDS y valores en [0,1]")

# ─── LEYENDA DE COLORES EN GEEMAP ─────────────────────

## Visualización del resultado en GEE / Geemap

Una vez generada la máscara densa con la U-Net, puedes subirla a Google Earth Engine
o visualizarla localmente con `rasterio` y `matplotlib`.

In [ ]:
# Visualización local con matplotlib (sin necesidad de GEE)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# --- Usa tu máscara real aquí ---
# Si tienes una imagen real exportada de GEE:
# import rasterio
# with rasterio.open('mi_imagen_completa.tif') as src:
#     img_np = src.read().astype(np.float32)
# for b in range(img_np.shape[0]):
#     mn, mx = img_np[b].min(), img_np[b].max()
#     img_np[b] = (img_np[b] - mn) / (mx - mn + 1e-8)
# pred_mask = predict_image_sliding_window(model, img_np)

# Demostración con máscara sintética
H_demo, W_demo = 256, 256
demo_mask = np.zeros((H_demo, W_demo), dtype=np.uint8)
demo_mask[50:120, 50:150] = 1   # Casas
demo_mask[160:200, 80:120] = 2  # Piscinas

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(mask_to_rgb(demo_mask))
ax.set_title('Mapa de Clasificación U-Net\n(0=Vegetación, 1=Casas, 2=Piscinas)', fontsize=13)
ax.axis('off')

legend_patches = [
    mpatches.Patch(color=[0, 180/255, 0],     label='0 — Vegetación'),
    mpatches.Patch(color=[220/255, 50/255, 50/255], label='1 — Casas/Edificios'),
    mpatches.Patch(color=[30/255, 100/255, 220/255], label='2 — Piscinas  ← nueva clase'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=11)
plt.tight_layout()
plt.savefig('mapa_clasificacion_unet.png', dpi=150)
plt.show()
print("Mapa guardado: mapa_clasificacion_unet.png")

# ─── COMPARACIÓN RF vs UNET ─────────────────────
---
# 📊 Comparación: Random Forest vs U-Net

| Característica          | Random Forest (original)     | U-Net (esta edición)            |
|-------------------------|------------------------------|---------------------------------|
| Tipo de modelo          | Árbol de decisión ensamblado | Red neuronal convolucional      |
| Unidad de análisis      | Píxel individual             | Parche espacial (64×64 px)      |
| Contexto espacial       | ❌ Ninguno                   | ✅ Encoder + Skip Connections   |
| Clases soportadas       | Verde, Casas *(Piscina ❌)*  | Verde, Casas, **Piscinas ✅**   |
| Entrenamiento           | GEE (en la nube)             | PyTorch (local/Colab)           |
| Tiempo de inferencia    | Segundos                     | Minutos (ventana deslizante)    |
| Escalabilidad           | Alta (GEE distribuido)       | Requiere GPU para imágenes grandes |
| Interpretabilidad       | Alta (feature importance)    | Menor (caja negra)              |
| Precisión esperada*     | ~75–80 % (3 clases)          | ~88–93 % (3 clases)             |

\\* Estimaciones orientativas sobre imágenes Sentinel-2 similares; dependen del número y calidad de puntos de entrenamiento.

## ¿Por qué la U-Net detecta piscinas y el RF no?

Las piscinas ocupan **pocos píxeles** en Sentinel-2 a 10 m/px. Su firma espectral
(alta reflectancia en B2-Azul, baja en B8-NIR) puede confundirse con sombras o
asfalto mojado cuando se analiza píxel a píxel. La U-Net usa el **contexto de los
píxeles vecinos** (a través de convoluciones y skip connections) para distinguir
la piscina de su entorno (bordes, forma rectangular, jardín circundante), lo que
la hace significativamente más precisa en objetos pequeños.

# ─── RESUMEN ───────────────────────────────────

# ✅ Resumen del flujo completo

1. **GEE Setup** — Autenticación, colección Sentinel-2, composición mediana.
2. **Etiquetado** — 3 clases (Verde=0, Casas=1, Piscinas=2) con puntos manuales.
3. **Índices** — NDVI, NDBI, NDWI añadidos a las bandas base (8 canales totales).
4. **Exportación** — Parches GeoTIFF de 64×64 px desde GEE a Google Drive.
5. **Dataset** — `SentinelPatchDataset` con normalización y data augmentation.
6. **U-Net** — Arquitectura con skip connections, 8 canales → 3 clases.
7. **Entrenamiento** — AdamW + Cosine Annealing + Loss = CrossEntropy + Dice.
8. **Evaluación** — IoU y F1 por clase (énfasis en clase Piscinas).
9. **Inferencia** — Ventana deslizante para imagen completa.
10. **Visualización** — Mapa de clasificación con leyenda de 3 colores.